In [ ]:
!pip install torch_geometric
!pip install networkx

In [ ]:
from pathlib import Path
import h5py

import networkx as nx
from sklearn.random_projection import GaussianRandomProjection


def load_graph_hdf5(h5_path: Path, graph_id):
    with h5py.File(h5_path, 'r') as h5f:
        grp = h5f[f'graph_{graph_id}']

        csr = sp.csr_matrix(
            (grp['adj_data'][:], grp['adj_indices'][:], grp['adj_indptr'][:]),
            shape=grp['adj_shape'][:]
        )


        valid_indices = grp['valid_indices'][:]
        local_valid_indices = grp['local_valid_indices']

        token_ids = grp['token_ids'][:]

        hidden_states = grp['hidden_states'][:]
        rp = GaussianRandomProjection(n_components=768, random_state=42)

        reduced_hidden_states = rp.fit_transform(hidden_states)

        G = nx.DiGraph()

        for i, (idx, token, hs) in enumerate(zip(valid_indices[local_valid_indices], token_ids, reduced_hidden_states)):
            G.add_node(int(idx), token=int(token), position=int(idx), embedding = hs)


        rows, cols = csr.nonzero()
        for row, col in zip(rows, cols):
            weight = csr[row, col]
            G.add_edge(int(valid_indices[local_valid_indices[col]]), int(valid_indices[local_valid_indices[row]]), weight=float(weight))

        return {
            'graph': G,
            'adj_matrix': csr,
            'valid_indices': valid_indices,
            'local_valid_indices': local_valid_indices,
            'token_ids': token_ids,
            'hidden_states': hidden_states,
            'num_nodes': grp.attrs['num_nodes'],
            'num_edges': grp.attrs['num_edges']
        }

def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def analyze_graph_simple(G):

    G_und = G.to_undirected()

    n_nodes = G_und.number_of_nodes()
    n_edges = G_und.number_of_edges()

    degree_centrality = nx.degree_centrality(G_und)

    node_size = [
      v * 1000 for v in degree_centrality.values()
    ]
    metrics = {}

    metrics['n_nodes'] = n_nodes
    metrics['n_edges'] = n_edges
    metrics['density'] = nx.density(G_und) if n_nodes > 0 else 0
    metrics['edge_to_node_ratio'] = n_edges / n_nodes if n_nodes > 0 else 0

    metrics['is_connected'] = int(nx.is_connected(G_und) if n_nodes > 0 else False)
    metrics['is_strongly_connected'] = int(nx.is_strongly_connected(G) if n_nodes > 0 else False)
    metrics['n_connected_components'] = nx.number_connected_components(G_und) if n_nodes > 0 else 0
    metrics['n_strongly_connected'] = nx.number_strongly_connected_components(G) if n_nodes > 0 else 0

    if n_nodes > 0:
        component_sizes = [len(c) for c in nx.connected_components(G_und)]
        metrics['largest_component_size'] = max(component_sizes) if component_sizes else 0
    else:
        metrics['largest_component_size'] = 0

    if n_nodes > 0:
        degree_seq = [d for n, d in G_und.degree()]
        in_degree_seq = [d for n, d in G.in_degree()]
        out_degree_seq = [d for n, d in G.out_degree()]

        metrics['avg_degree'] = np.mean(degree_seq)
        metrics['median_degree'] = np.median(degree_seq)
        metrics['std_degree'] = np.std(degree_seq)
        metrics['min_degree'] = min(degree_seq)
        metrics['max_degree'] = max(degree_seq)
        metrics['degree_variance'] = np.var(degree_seq)
        metrics['avg_in_degree'] = np.mean(in_degree_seq) if in_degree_seq else 0
        metrics['avg_out_degree'] = np.mean(out_degree_seq) if out_degree_seq else 0
        metrics['degree_assortativity'] = nx.degree_assortativity_coefficient(G_und)

        probs = np.bincount(degree_seq) / n_nodes
        probs = probs[probs > 0]
        metrics['degree_entropy'] = -np.sum(probs * np.log2(probs)) if len(probs) > 0 else 0
    else:
        metrics.update({
            'avg_degree': 0, 'median_degree': 0, 'std_degree': 0, 'min_degree': 0,
            'max_degree': 0, 'degree_variance': 0, 'avg_in_degree': 0, 'avg_out_degree': 0,
            'degree_assortativity': 0, 'degree_entropy': 0
        })

    if n_nodes > 0:
        metrics['avg_clustering'] = nx.average_clustering(G_und)
        metrics['transitivity'] = nx.transitivity(G_und)
        metrics['n_triangles'] = sum(nx.triangles(G_und).values()) // 3
    else:
        metrics['avg_clustering'] = 0
        metrics['transitivity'] = 0
        metrics['n_triangles'] = 0

    if nx.is_connected(G_und) and n_nodes > 1:
        metrics['diameter'] = nx.diameter(G_und)
        metrics['radius'] = nx.radius(G_und)
        metrics['avg_path_length'] = nx.average_shortest_path_length(G_und)

        ecc = nx.eccentricity(G_und)
        metrics['avg_eccentricity'] = np.mean(list(ecc.values()))
        metrics['max_eccentricity'] = max(ecc.values())
        metrics['min_eccentricity'] = min(ecc.values())
    else:
        if n_nodes > 0:
            largest_cc = max(nx.connected_components(G_und), key=len)
            subgraph = G_und.subgraph(largest_cc)
            if len(subgraph) > 1:
                metrics['diameter'] = nx.diameter(subgraph)
                metrics['radius'] = nx.radius(subgraph)
                metrics['avg_path_length'] = nx.average_shortest_path_length(subgraph)
                ecc = nx.eccentricity(subgraph)
                metrics['avg_eccentricity'] = np.mean(list(ecc.values()))
                metrics['max_eccentricity'] = max(ecc.values())
                metrics['min_eccentricity'] = min(ecc.values())
            else:
                metrics.update({'diameter': 0, 'radius': 0, 'avg_path_length': 0,
                               'avg_eccentricity': 0, 'max_eccentricity': 0, 'min_eccentricity': 0})
        else:
            metrics.update({'diameter': 0, 'radius': 0, 'avg_path_length': 0,
                           'avg_eccentricity': 0, 'max_eccentricity': 0, 'min_eccentricity': 0})

    try:
        if n_nodes > 0:
            has_loop = not nx.is_forest(G_und) if n_nodes > 0 else False

            if has_loop:
                n_components = nx.number_connected_components(G_und)
                loop_count = n_edges - n_nodes + n_components
            else:
                loop_count = 0
        else:
            has_loop = False
            loop_count = 0
    except:
        has_loop = False
        loop_count = 0

    metrics['has_loop'] = has_loop
    metrics['loop_count'] = loop_count

    if n_nodes > 1 and metrics['avg_clustering'] > 0 and metrics['avg_path_length'] > 0:

        N = n_nodes
        K = metrics['avg_degree']

        if K > 1 and N > 1:
            C_rand = K / (N - 1) if N > 1 else 0
            L_rand = np.log(N) / np.log(K) if K > 1 else float('inf')
            clustering_norm = metrics['avg_clustering'] / C_rand if C_rand > 0 else 0
            path_length_norm = metrics['avg_path_length'] / L_rand if L_rand not in [0, float('inf')] else 0

            small_world_index = clustering_norm / path_length_norm if path_length_norm > 0 else 0
        else:
            small_world_index = 0
    else:
        small_world_index = 0

    metrics['small_world_index'] = small_world_index

    if n_nodes > 0:
        deg_cent = nx.degree_centrality(G_und)
        metrics['avg_degree_centrality'] = np.mean(list(deg_cent.values()))
        metrics['max_degree_centrality'] = max(deg_cent.values())
        metrics['std_degree_centrality'] = np.std(list(deg_cent.values()))

        in_deg_cent = nx.in_degree_centrality(G)
        out_deg_cent = nx.out_degree_centrality(G)
        metrics['avg_in_degree_centrality'] = np.mean(list(in_deg_cent.values())) if in_deg_cent else 0
        metrics['avg_out_degree_centrality'] = np.mean(list(out_deg_cent.values())) if out_deg_cent else 0

        bet_cent = nx.betweenness_centrality(G_und)
        metrics['avg_betweenness'] = np.mean(list(bet_cent.values()))
        metrics['max_betweenness'] = max(bet_cent.values())
        metrics['std_betweenness'] = np.std(list(bet_cent.values()))

        if any('weight' in d for (_, _, d) in G.edges(data=True)):
            bet_cent_weighted = nx.betweenness_centrality(G_und, weight='weight')
            metrics['avg_betweenness_weighted'] = np.mean(list(bet_cent_weighted.values()))
            metrics['max_betweenness_weighted'] = max(bet_cent_weighted.values())
        else:
            metrics['avg_betweenness_weighted'] = metrics['avg_betweenness']
            metrics['max_betweenness_weighted'] = metrics['max_betweenness']

        if nx.is_connected(G_und):
            clo_cent = nx.closeness_centrality(G_und)
        else:
            clo_cent = nx.harmonic_centrality(G_und)
        metrics['avg_closeness'] = np.mean(list(clo_cent.values()))
        metrics['max_closeness'] = max(clo_cent.values())


        try:
            eig_cent = nx.eigenvector_centrality(G_und, max_iter=1000)
            metrics['avg_eigenvector'] = np.mean(list(eig_cent.values()))
            metrics['max_eigenvector'] = max(eig_cent.values())
        except:
            metrics['avg_eigenvector'] = 0
            metrics['max_eigenvector'] = 0

        try:
            katz_cent = nx.katz_centrality(G_und, alpha=0.1)
            metrics['avg_katz'] = np.mean(list(katz_cent.values()))
            metrics['max_katz'] = max(katz_cent.values())
        except:
            metrics['avg_katz'] = 0
            metrics['max_katz'] = 0

    else:
        null_cent = ['avg_degree_centrality', 'max_degree_centrality', 'std_degree_centrality',
                    'avg_in_degree_centrality', 'avg_out_degree_centrality', 'avg_betweenness',
                    'max_betweenness', 'std_betweenness', 'avg_betweenness_weighted', 'max_betweenness_weighted',
                    'avg_closeness', 'max_closeness', 'avg_eigenvector', 'max_eigenvector',
                    'avg_katz', 'max_katz']
        for c in null_cent:
            metrics[c] = 0

    return G, metrics

In [ ]:

import json
import pandas as pd
import scipy.sparse as sp
import pickle
from tqdm import tqdm
import numpy as np


traces_path = '/content/traces_math500_deepseek32b_vllm_final.jsonl'
h5_path = '/content/attention_graphs_with_hidden_5120_topk2_20_nodes_math500_DeepSeek32B.h5'

traces = []

features_path = "attention_graph_DeepSeek32B_math_all_graph_features.jsonl"

with open(traces_path, 'r') as f:
    for line in f:
      if line.strip():
          trace = json.loads(line)
          traces.append(trace)

traces_df = pd.DataFrame(traces)
traces_df = traces_df.dropna(subset=['trace'])
traces_df = traces_df[traces_df['trace'] != '']

with open(features_path, 'w') as out_f:
  for id in tqdm(enumerate(traces_df['id'].tolist())):
    graph_data = load_graph_hdf5(h5_path, id[1])
    path = id[1].split('.')[0].replace('/', '_')
    graph_path = Path('graph_dir')/ f"{path}.pkl"
    G = graph_data['graph']
    ensure_dir(graph_path)
    with open(graph_path, 'wb') as f:
        pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

    G, metrics = analyze_graph_simple(G)
    row = {
            "id": id[1],
              **metrics
          }
    out_f.write(json.dumps(row) + "\n")

500it [01:31,  5.44it/s]


In [ ]:
import json
import pandas as pd
import scipy.sparse as sp
import pickle
from tqdm import tqdm
from pathlib import Path
import networkx as nx

def ensure_dir(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)

traces_path = '/content/traces_math500_deepseek32b_vllm_final.jsonl'
gml_path = '/content/graph_dir'

traces = []

with open(traces_path, 'r') as f:
    for line in f:
      if line.strip():
          trace = json.loads(line)
          traces.append(trace)

traces_df = pd.DataFrame(traces)

for id in tqdm(enumerate(traces_df['id'].tolist())):
  gml_path = Path('/content/graph_dir') / f'{id[1].split('.')[0].replace('/', '_')}.json.graphml'
  graph_data = G = nx.read_gml(gml_path)
  graph_path = Path('pickle_graph_dir')/ f'{id[1].split('.')[0].replace('/', '_')}.pkl'
  ensure_dir(graph_path)
  with open(graph_path, 'wb') as f:
      pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

500it [01:01,  8.13it/s]


In [ ]:
!zip -r global_graphs_pickle_200_clusters_math_DeepSeek2.5_32B.zip pickle_graph_dir

In [ ]:
!zip -r attention_graphs_with_hidden_768_rp_pickle_max_token_20_topk2_math_Qwen2.5_1.5B.zip graph_dir

In [ ]:
from google.colab import files

files.download('/content/attention_graph_Qwen32B_math_all_graph_features.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool

class GNNBinaryClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_weight, batch):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        graph_embedding = global_mean_pool(x, batch)
        out = self.classifier(graph_embedding)
        return out

In [ ]:
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data as PyGData
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from sklearn.metrics.pairwise import cosine_distances
import networkx as nx
import os
import json
import pandas as pd
import numpy as np
import pickle

class GraphBinaryDataset(Dataset):

    def __init__(self, dataset, graph_dir, traces_path, preload=False, transform=None):
        self.dataset = dataset
        self.graph_dir = graph_dir
        self.transform = transform
        self.preload = preload

        self.graph_files = [f for f in os.listdir(graph_dir)
                           if f.endswith(('.graphml')) or f.endswith(('.pkl')) ]

        print(self.graph_files)


        if os.path.exists(traces_path):
            traces = []
            with open(traces_path, 'r') as f:
               for line in f:
                  if line.strip():
                      trace = json.loads(line)
                      traces.append(trace)

            traces_df = pd.DataFrame(traces)
            if self.dataset == 'math500':
              traces_df['id'] = traces_df['id'].apply(lambda x: x.split('.')[0].replace('/', '_'))
            traces_df['is_correct'] = traces_df.apply(lambda x: 'Yes' if (x.is_correct == 'Yes') else 'No', axis = 1)
            traces_df['is_correct'] = traces_df['is_correct'].apply(lambda x: 0.0 if x == 'Yes' else 1.0)
            self.traces = traces_df[['id', 'is_correct']]

        if preload:
            print("Preloading all graphs into memory...")
            self.data = []
            for idx in range(len(self.graph_files)):
                if idx % 100 == 0:
                    print(f"  Loading {idx}/{len(self.graph_files)}")
                pyg_data, label = self._load_item(idx)
                self.data.append((pyg_data, label))
            print(f"Preloaded {len(self.data)} graphs")


    def __len__(self):
        return len(self.graph_files)

    def _load_item(self, idx):
        graph_file = self.graph_files[idx]
        graph_path = os.path.join(self.graph_dir, graph_file)

        if self.dataset == 'math500':
            id = graph_file.split('.')[0]
            label = self.traces[self.traces.id == id]['is_correct'].values[0]
        else:
            id = graph_path.split('.')[0].split('/')[1]
            label = self.traces[self.traces.id == int(id)]['is_correct'].values[0]

        if graph_file.endswith('.graphml'):
          G = nx.read_gml(graph_path)
        if graph_file.endswith('.pkl'):
          with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        pyg_data = self._nx_to_pyg(G, edge_metric='cosine')

        if isinstance(label, (int, float)):
            label = torch.tensor([label], dtype=torch.float32)

        return pyg_data, label

    def __getitem__(self, idx):
        if self.preload:
            pyg_data, label = self.data[idx]
            return pyg_data, label.clone() if isinstance(label, torch.Tensor) else label
        else:
            return self._load_item(idx)

    def _nx_to_pyg(self, G, edge_metric=None):

        nodes = list(G.nodes())
        node_to_idx = {node: i for i, node in enumerate(nodes)}


        node_features = []
        for node in nodes:
            if 'embedding' in G.nodes[node]:
                feat = np.array(G.nodes[node]['embedding'])

            node_features.append(feat)


        node_features = np.array(node_features)

        x = torch.tensor(node_features, dtype=torch.float)

        edges = []
        edge_weights = []

        for u, v in G.edges():

            edges.append([node_to_idx[u], node_to_idx[v]])
            if edge_metric:
              weight = cosine_distances([G.nodes[u]['embedding']], [G.nodes[v]['embedding']])[0][0]
            else:
              weight = G[u][v].get('weight', 1.0)
            edge_weights.append(weight)

            if not G.is_directed():
                edges.append([node_to_idx[v], node_to_idx[u]])
                edge_weights.append(weight)

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_weight = torch.tensor(edge_weights, dtype=torch.float) if edge_weights else None

        pyg_data = PyGData(x=x, edge_index=edge_index, edge_weight=edge_weight)

        return pyg_data



def main():
    GRAPH_DIR = 'pickle_graph_dir'
    traces_path = '/content/traces_math500_deepseek32b_vllm_final.jsonl'

    dataset = GraphBinaryDataset('math500', GRAPH_DIR, traces_path)
    print(len(dataset))
    train_size = int(0.7 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size

    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )

    BATCH_SIZE = 32

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2
    )

    return train_loader, val_loader, test_loader


if __name__ == "__main__":
    train_loader, val_loader, test_loader = main()

['test_algebra_1553.pkl', 'test_counting_and_probability_1014.pkl', 'test_intermediate_algebra_1555.pkl', 'test_precalculus_1291.pkl', 'test_precalculus_986.pkl', 'test_geometry_283.pkl', 'test_counting_and_probability_10.pkl', 'test_prealgebra_1865.pkl', 'test_prealgebra_1991.pkl', 'test_counting_and_probability_666.pkl', 'test_precalculus_477.pkl', 'test_prealgebra_1655.pkl', 'test_algebra_518.pkl', 'test_algebra_478.pkl', 'test_geometry_802.pkl', 'test_algebra_864.pkl', 'test_geometry_686.pkl', 'test_prealgebra_1622.pkl', 'test_prealgebra_153.pkl', 'test_algebra_109.pkl', 'test_algebra_2680.pkl', 'test_intermediate_algebra_2196.pkl', 'test_geometry_1097.pkl', 'test_algebra_2080.pkl', 'test_algebra_776.pkl', 'test_prealgebra_2086.pkl', 'test_number_theory_255.pkl', 'test_intermediate_algebra_1797.pkl', 'test_number_theory_234.pkl', 'test_algebra_2391.pkl', 'test_algebra_170.pkl', 'test_geometry_248.pkl', 'test_precalculus_96.pkl', 'test_algebra_893.pkl', 'test_counting_and_probabilit

In [ ]:
import numpy as np

def calculate_metrics(outputs, labels):
    probs = torch.sigmoid(outputs).detach().cpu().numpy().flatten()
    preds = (probs > 0.5).astype(int)
    print(probs.mean())

    labels_np = labels.detach().cpu().numpy().flatten()
    outputs = outputs.detach().cpu().numpy()

    print(np.unique(preds))

    metrics = {}

    metrics['accuracy'] = accuracy_score(labels_np, preds)

    metrics['recall'] = recall_score(labels_np, preds, zero_division=0)

    metrics['roc_auc'] = roc_auc_score(labels_np, outputs)

    metrics['pr_auc'] = average_precision_score(labels_np, outputs)

    return metrics

In [ ]:
def evaluate(model, data_loader, criterion, device='cpu'):

    model.eval()
    total_loss = 0
    all_outputs = []
    all_labels = []

    with torch.no_grad():
        for graphs, labels in data_loader:

            graphs = graphs.to(device)
            labels = labels.to(device)

            outputs = model(graphs.x, graphs.edge_index, graphs.edge_weight, graphs.batch)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            all_outputs.append(outputs)
            all_labels.append(labels)


    all_outputs = torch.cat(all_outputs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = calculate_metrics(all_outputs, all_labels)
    avg_loss = total_loss / len(data_loader)

    return avg_loss, metrics

In [ ]:
!pip install early-stopping-pytorch

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, average_precision_score
from early_stopping_pytorch import EarlyStopping

early_stopping = EarlyStopping(patience=7, verbose=True)

model = GNNBinaryClassifier(input_dim=768, hidden_dim=64)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

losses = []

val_losses = []
num_epochs = 10


iter = 0
while True:

  all_train_outputs = []
  all_train_labels = []
  losses = []

  for batch_idx, (graphs, labels) in enumerate(train_loader):

      outputs = model(graphs.x, graphs.edge_index, graphs.edge_weight, graphs.batch)
      loss = criterion(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      all_train_outputs.append(outputs)
      all_train_labels.append(labels)

      losses.append(loss.item())

  all_train_outputs = torch.cat(all_train_outputs, dim=0)
  all_train_labels = torch.cat(all_train_labels, dim=0)

  mean_loss = np.mean(np.array(losses))
  train_metrics = calculate_metrics(all_train_outputs, all_train_labels)

  print(f"Epoch {iter+1}/{num_epochs}")
  print(f"{'='*60}")
  print(f"Train Loss: {mean_loss:.4f}")
  print(f"\nTrain Metrics:")
  print(f"  Accuracy: {train_metrics['accuracy']:.4f}")
  print(f"  Recall:   {train_metrics['recall']:.4f}")
  print(f"  ROC-AUC:  {train_metrics['roc_auc']:.4f}")
  print(f"  PR-AUC:   {train_metrics['pr_auc']:.4f}")

  val_loss, val_metrics = evaluate(model, val_loader, criterion)
  val_losses.append(val_loss)

  print(f"\nVal Metrics:")
  print(f"  Accuracy: {val_metrics['accuracy']:.4f}")
  print(f"  Recall:   {val_metrics['recall']:.4f}")
  print(f"  ROC-AUC:  {val_metrics['roc_auc']:.4f}")
  print(f"  PR-AUC:   {val_metrics['pr_auc']:.4f}")
  print(f"{'='*60}\n")

  iter += 1

  early_stopping(val_loss, model)
  if early_stopping.early_stop:
      print("Early stopping triggered")
      break

0.46653897
[0 1]
Epoch 1/10
Train Loss: 0.6542

Train Metrics:
  Accuracy: 0.7086
  Recall:   0.2258
  ROC-AUC:  0.4897
  PR-AUC:   0.1763
0.39076483
[0]

Val Metrics:
  Accuracy: 0.8000
  Recall:   0.0000
  ROC-AUC:  0.7275
  PR-AUC:   0.4259

Validation loss decreased (inf --> 0.576238).  Saving model ...
0.2990365
[0]
Epoch 2/10
Train Loss: 0.5027

Train Metrics:
  Accuracy: 0.8229
  Recall:   0.0000
  ROC-AUC:  0.5644
  PR-AUC:   0.2230
0.18757625
[0]

Val Metrics:
  Accuracy: 0.8000
  Recall:   0.0000
  ROC-AUC:  0.7675
  PR-AUC:   0.4700

Validation loss decreased (0.576238 --> 0.472636).  Saving model ...
0.13846712
[0]
Epoch 3/10
Train Loss: 0.4487

Train Metrics:
  Accuracy: 0.8229
  Recall:   0.0000
  ROC-AUC:  0.7024
  PR-AUC:   0.3490
0.112982
[0]

Val Metrics:
  Accuracy: 0.8000
  Recall:   0.0000
  ROC-AUC:  0.7875
  PR-AUC:   0.4914

EarlyStopping counter: 1 out of 7
0.13284078
[0]
Epoch 4/10
Train Loss: 0.4346

Train Metrics:
  Accuracy: 0.8229
  Recall:   0.0000
  ROC-

In [ ]:
test_loss, test_metrics = evaluate(model, test_loader, criterion)
print("TEST SET RESULTS:")
for metric, value in test_metrics.items():
    print(f"  {metric.upper()}: {value:.4f}")

0.14527032
[0 1]
TEST SET RESULTS:
  ACCURACY: 0.9000
  RECALL: 0.5000
  ROC_AUC: 0.9092
  PR_AUC: 0.8151
